# Bridge Design Pattern 

explained using the classic Remote Control and Device example.

#### The Concept

The Bridge Pattern splits a large class hierarchy into two separate hierarchies:
- Abstraction: The "Control" layer (The Remote).
- Implementation: The "Device" layer (The TV or Radio).

**Abstraction** (The Interface/Control) and **Implementation** (The Platform/Engine). These two are connected by a **"Bridge" (composition)**

**Why?** Without the Bridge, if you have 2 types of remotes **(Basic, Advanced)** and 2 types of devices (TV, Radio), you might end up with 4 classes: `BasicTvRemote`, `BasicRadioRemote`, `AdvancedTvRemote`, `AdvancedRadioRemote`. This grows exponentially (Cartesian Product complexity). With the Bridge, the Remote simply holds a reference to a Device. You can mix and match them freely.

**The Solution**:
- Hierarchy 1: Remotes (Holds a reference to a Device).
- Hierarchy 2: Devices (Implements the commands). You can now mix and match any Remote with any Device.

## The Classic OOP Way (Java-Style)

We define the `Device` interface (Implementation) and the `Remote` class (Abstraction). The Remote holds a reference to `Device`.

#### IMPLEMENTATION LAYER ( The Device )

In [12]:
from abc import ABC, abstractmethod

class Device(ABC):
    @abstractmethod
    def is_enabled(self) -> bool:pass
    
    @abstractmethod
    def enable(self): pass

    @abstractmethod
    def disable(self): pass

    @abstractmethod
    def set_volume(self, percent: int): pass

#### Concrete Implementations

In [13]:
class TV(Device):
    def __init__(self):
        self.on = False
        self.volume = 30

    def is_enabled(self) -> bool: return self.on
    def enable(self): 
        self.on = True
        print("TV: Turned ON")
        
    def disable(self): 
        self.on = False
        print("TV: Turned OFF")
        
    def set_volume(self, percent: int):
        self.volume = percent
        print(f"TV: Volume set to {self.volume}")

class Radio(Device):
    def __init__(self):
        self.on = False
        self.volume = 10

    def is_enabled(self) -> bool: return self.on
    def enable(self): 
        self.on = True
        print("Radio: Turned ON")
        
    def disable(self): 
        self.on = False
        print("Radio: Turned OFF")
        
    def set_volume(self, percent: int):
        self.volume = percent
        print(f"Radio: Volume set to {self.volume}")

#### ABSTRACTION LAYER ( The Remote )

In [14]:
class RemoteControl:
    """
    The Abstraction. It holds a reference (the 'bridge') to the Device.
    It delegates the actual work to the device object.
    """
    def __init__(self, device: Device):
        self.device = device  # <--- THE BRIDGE

    def toggle_power(self):
        if self.device.is_enabled():
            self.device.disable()
        else:
            self.device.enable()

    def volume_down(self):
        print("Remote: Volume Down")
        self.device.set_volume(0) # Logic simplified

# --- Refined Abstraction ---
class AdvancedRemoteControl(RemoteControl):
    """
    Extends the Remote functionality without changing the Device classes.
    """
    def mute(self):
        print("Remote: Mute Button Pressed")
        self.device.set_volume(0)

#### CLIENT CODE

In [15]:
def main():
    tv = TV()
    radio = Radio()

    # We can pair ANY remote with ANY device
    remote_tv = RemoteControl(tv)
    remote_tv.toggle_power()

    print("-" * 20)

    # Advanced remote works with Radio too
    advanced_remote_radio = AdvancedRemoteControl(radio)
    advanced_remote_radio.toggle_power()
    advanced_remote_radio.mute()

if __name__ == "__main__":
    main()

TV: Turned ON
--------------------
Radio: Turned ON
Remote: Mute Button Pressed
Radio: Volume set to 0


## The Pythonic Way

In Python, we can simplify this using **Composition** and **Protocols (Duck Typing)**. We don't need rigid inheritance for the implementation layer (TV doesn't need to inherit from Device) and strict `Abstract Base Class` for the implementation if the methods align.. As long as it has the methods, the bridge works.

Ideally, the Bridge pattern is just "Dependency Injection" applied to a class hierarchy.

#### PROTOCOL (Implicit Interface)

In [16]:
from typing import Protocol

class Device(Protocol):
    def turn_on(self) -> None: ...
    def turn_off(self) -> None: ...
    def set_channel(self, channel: int) -> None: ...

#### CONCRETE DEVICES (No Inheritance needed)

In [17]:
class TV:
    def turn_on(self): print("TV: ON")
    def turn_off(self): print("TV: OFF")
    def set_channel(self, channel): print(f"TV: Channel {channel}")

class Radio:
    def turn_on(self): print("Radio: ON")
    def turn_off(self): print("Radio: OFF")
    def set_channel(self, channel): print(f"Radio: Frequency {channel}")

#### THE BRIDGE (The Remotes)

In [18]:
class BaseRemote:
    def __init__(self, device: Device):
        self.device = device # The Bridge

    def power(self):
        print("Remote: Power button pressed.")
        self.device.turn_on() # Simple delegation

class AdvancedRemote(BaseRemote):
    def mute(self):
        print("Remote: Mute.")
        # We can add logic here without touching the TV class
        self.device.set_channel(0)

#### CLIENT CODE

In [19]:
def main():
    # Mix and Match
    my_tv = TV()
    my_radio = Radio()

    # Bridge: Connect Advanced Remote -> TV
    remote = AdvancedRemote(my_tv)
    remote.power()
    remote.mute()

    print("--- Switching Device ---")

    # Bridge: Reuse SAME Remote logic -> Radio
    # We swapped the implementation at runtime!
    remote.device = my_radio
    remote.power()

if __name__ == "__main__":
    main()

Remote: Power button pressed.
TV: ON
Remote: Mute.
TV: Channel 0
--- Switching Device ---
Remote: Power button pressed.
Radio: ON


| Key Differences | Feature      | Classic OOP                                   | Pythonic                                        |
|-----------------|--------------|-----------------------------------------------|-------------------------------------------------|
| Device Layer    | Implementation | Inherits from an abstract `Device` class.     | Uses `Protocol` (duck typing). TV is standalone. |
| Coupling        | Dependency   | Tighter coupling via inheritance.             | Loose coupling via structural typing.           |
| Flexibility     | Design       | High flexibility, but requires more setup code. | Very high flexibility with minimal boilerplate.  |


### Summary

The Bridge pattern is essentially Composition over Inheritance taken to the architectural level. Instead of class TvRemote(Remote), you have class Remote which has a TV.

# Bridge Design Pattern 

explained using a complex, real-world scenario: A Multi-Channel Notification System.

##### The Scenario: Enterprise Messaging Platform

You are building a system that sends notifications to users. **Complexity 1 (The Abstraction)**: You have different types of messages with different business logic.
- **Simple Notification**: Just sends the message.
- **Urgent Notification**: Sends, waits for an acknowledgement, and retries if it fails.
- **Digest Notification**: Buffers messages and sends them in a batch.

**Complexity 2 (The Implementation)**: You have different channels (vendors) to send these messages.
- Email (e.g., SendGrid).
- SMS (e.g., Twilio).
- Slack (Webhook).

**The Problem**: If you use inheritance, you get a Cartesian Product explosion: `UrgentEmail`, `UrgentSMS`, `SimpleEmail`, `SimpleSMS`, `DigestEmail`... If you add "WhatsApp", you have to create 3 new classes. 

**The Solution**: Isolate "How it behaves" (Abstraction) from "How it travels" (Implementation).

## The Classic OOP Way (Java-Style)

We define an interface `IMessageSender` for the vendors. The `Notification` abstract class holds a reference to this interface (The Bridge).

#### THE IMPLEMENTATION INTERFACE (The Platform)

In [20]:
from abc import ABC, abstractmethod

class IMessageSender(ABC):
    @abstractmethod
    def send_message(self, subject: str, body: str):
        pass

#### CONCRETE IMPLEMENTATIONS (Vendors)

In [21]:
class EmailSender(IMessageSender):
    def send_message(self, subject: str, body: str):
        print(f"📧 [SendGrid] Sending Email: '{subject}' -> {body}")

class SMSSender(IMessageSender):
    def send_message(self, subject: str, body: str):
        print(f"📱 [Twilio] Sending SMS: {body} (Subject: {subject})")

class SlackSender(IMessageSender):
    def send_message(self, subject: str, body: str):
        print(f"💬 [Slack] Posting to #general: *{subject}* - {body}")

#### THE ABSTRACTION (The Business Logic)

In [22]:
class Notification(ABC):
    def __init__(self, sender: IMessageSender):
        self.sender = sender # THE BRIDGE

    @abstractmethod
    def notify(self, message: str):
        pass

#### REFINED ABSTRACTIONS (Specific Behaviors)

In [23]:
class SimpleNotification(Notification):
    def notify(self, message: str):
        # Just pass it through
        self.sender.send_message("Info", message)

class UrgentNotification(Notification):
    def notify(self, message: str):
        print("🚨 [Urgent] Processing priority message...")
        # Urgent logic: Send with a visual flag (simulated by subject)
        self.sender.send_message("URGENT ALERT", message.upper())
        # Logic: We could add retry logic here, which applies to ANY sender
        print("🚨 [Urgent] Verifying delivery status...")

#### CLIENT CODE

In [24]:
def main():
    # 1. Pick a Platform (Implementation)
    email = EmailSender()
    sms = SMSSender()

    # 2. Pick a Behavior (Abstraction)
    # Notice: UrgentNotification works with SMS without code changes!
    
    n1 = UrgentNotification(sms) 
    n1.notify("Server is down!")

    print("-" * 20)

    n2 = SimpleNotification(email)
    n2.notify("Daily Report Ready")

if __name__ == "__main__":
    main()

🚨 [Urgent] Processing priority message...
📱 [Twilio] Sending SMS: SERVER IS DOWN! (Subject: URGENT ALERT)
🚨 [Urgent] Verifying delivery status...
--------------------
📧 [SendGrid] Sending Email: 'Info' -> Daily Report Ready


## The Pythonic Way (Protocols & Dependency Injection)

In Python, we can loosen the constraints. We don't need strict abstract classes for the vendors; we just need objects that satisfy the **Protocol** (Duck Typing). The "Bridge" is simply the `sender` argument passed during initialization.

We can also treat the "Implementation" as a `Function` if the implementation is simple (e.g., just a function that calls an API), further simplifying the code.

#### THE PROTOCOL (Contract for Vendors)

In [25]:
from typing import Protocol

class SenderProtocol(Protocol):
    def send(self, title: str, content: str) -> None: ...

#### THE IMPLEMENTATIONS (Clean Classes)

In [26]:
class TwilioGateway:
    def send(self, title: str, content: str):
        print(f"📱 Twilio SMS: [{title}] {content}")

class SlackGateway:
    def send(self, title: str, content: str):
        print(f"💬 Slack Bot: {title} -> {content}")

#### THE ABSTRACTION (Bridge Logic)

In [28]:
from dataclasses import dataclass

@dataclass
class SystemAlert:
    """
    Standard Logic: Just sends the message.
    """
    gateway: SenderProtocol # The Bridge

    def trigger(self, msg: str):
        self.gateway.send("System Info", msg)

class CriticalRetryAlert(SystemAlert):
    """
    Complex Logic: Retries 3 times if it fails.
    This logic applies regardless of whether it's Slack or SMS.
    """
    def trigger(self, msg: str):
        print(f"🚨 CRITICAL ALERT STARTING...")
        for i in range(1, 4):
            try:
                print(f"   (Attempt {i}) sending via {type(self.gateway).__name__}...")
                self.gateway.send("CRITICAL", msg)
                print("   ✅ Success")
                return
            except Exception:
                print("   ❌ Failed, retrying...")
        print("   💀 All retries failed.")

#### CLIENT CODE

In [29]:
def main():
    # 1. Setup the Implementations
    sms = TwilioGateway()
    slack = SlackGateway()

    # 2. Mix and Match
    print("--- Scenario A: Info via Slack ---")
    # Bridge established by passing 'slack' into 'SystemAlert'
    alert = SystemAlert(slack)
    alert.trigger("Backup started")

    print("\n--- Scenario B: Critical Error via SMS ---")
    # Bridge established by passing 'sms' into 'CriticalRetryAlert'
    # The Critical logic (retries) is now applied to the SMS channel
    critical = CriticalRetryAlert(sms)
    critical.trigger("Database DB-01 is unreachable")

if __name__ == "__main__":
    main()

--- Scenario A: Info via Slack ---
💬 Slack Bot: System Info -> Backup started

--- Scenario B: Critical Error via SMS ---
🚨 CRITICAL ALERT STARTING...
   (Attempt 1) sending via TwilioGateway...
📱 Twilio SMS: [CRITICAL] Database DB-01 is unreachable
   ✅ Success


#### Why this is a powerful pattern in Industry

- **Vendor Lock-in Avoidance**: You can switch from `Twilio` to `AWS` `SNS` by just writing a new `AWSGateway` class. You do not have to touch the complex `CriticalRetryAlert` logic.
- **Testability**: You can easily inject a **MockGateway** into **CriticalRetryAlert** to test the retry logic without actually sending real SMS messages and incurring costs.
- Orthogonal Scaling:
    - If you add a new Channel (WhatsApp), you write 1 class.
    - If you add a new Logic (Digest/Batching), you write 1 class.
    - Without Bridge, adding WhatsApp would force you to write `WhatsAppSimple`, `WhatsAppUrgent`, `WhatsAppDigest`, etc.